# exp_3 — train on the competition's possession definition (pre-registered in `PREREG.md`)

**Question.** exp_1/exp_2 learn IDSSE's provider possession field (4.58 possessions/min, including
touch-level spells). CHALLENGE.md asks for *controlled* possession: a touch, a deflection or a won aerial
duel does not change possession. Does SetSeq trained, calibrated and decoded on a controlled-possession
relabel (`L_ctrl`) beat the provider-trained ensemble re-decoded on `L_ctrl` (H1)? And how do the
conclusions move under other label definitions (S1) and with more ball coverage (S2)?

**`L_ctrl`.** A team run is a controlled possession if it starts the half or follows a dead ball, lasts
≥ 2 s, or (if the event sync passes its pre-registered check) contains a controlled on-ball event;
otherwise it joins the previous team. The event sync failed its check, so `L_ctrl` = rules (a) + (b),
which equals `L_track(2 s)` (PREREG §2.3).

**What is new vs exp_0/exp_1/exp_2/rasya/ryu.**

| component | exp_3 | closest prior work |
|---|---|---|
| training / selection labels | `L_ctrl` (controlled possession, CHALLENGE.md) | exp_0–exp_2: provider `BallPossession`; ryu: event-derived possessions |
| label sensitivity (S1) | every arm against provider, `L_track(1/2/3 s)`, `L_ctrl`, re-selected per label set | none |
| coverage sensitivity (S2) | clean ball re-revealed on 12 % of hidden frames, saved weights | none |
| fusion | exp_0's geometric acceptance + a replay guard | exp_0: acceptance only |


In [1]:
from pathlib import Path
import os, sys, json, time
ROOT = Path.cwd()
if not (ROOT / 'exp3_common.py').exists():
    raise RuntimeError('Run with working directory exp_jeremy/exp_3')
sys.path.insert(0, str(ROOT))
sys.dont_write_bytecode = True
import numpy as np
import pandas as pd
pd.set_option('display.width', 220)
pd.set_option('display.max_columns', 40)
import exp3_common as C
import exp1_common as C1
if not C.PROTOCOL.exists():
    C.freeze_prereg()
    print('PREREG frozen now (before any exp_3 fit)')
protocol = C.verify_prereg()
print('PREREG verified', protocol['prereg_sha256'][:16], '| frozen', protocol['frozen_at_utc'])
import exp3_protocol as P, exp3_diagnostics as G, exp3_labels as L

PREREG frozen now (before any exp_3 fit)
PREREG verified cff444fff0106e9c | frozen 2026-11-08T16:53:50+00:00


## 1. Label integrity
Relabelings may only move team runs: D frames and frames outside the halves are unchanged. Under the fallback, `L_ctrl` must equal `L_track(2 s)` on every match.

In [2]:
integrity = pd.DataFrame(P.label_integrity())
C1.write_json(C.RESULTS / 'label_integrity.json', integrity.to_dict(orient='records'))
print(integrity.to_string(index=False))
assert integrity.drop(columns='match').all().all()

 match  ctrl_dead_untouched  ctrl_outside_halves_untouched  track_1_dead_untouched  track_1_outside_halves_untouched  track_2_dead_untouched  track_2_outside_halves_untouched  track_3_dead_untouched  track_3_outside_halves_untouched  ctrl_equals_track_2  degraded_labels_equal_clean
J03WMX                 True                           True                    True                              True                    True                              True                    True                              True                 True                         True
J03WN1                 True                           True                    True                              True                    True                              True                    True                              True                 True                         True
J03WOH                 True                           True                    True                              True                    True           

## 2. Event sync (IDSSE open event data)
Kick-off sync, then label-free snap to the acting player's closest approach to the ball. Acceptance (PREREG §2.3): pooled shot residual against the provider's `CalculatedFrame`, median |r| ≤ 5 and p90 |r| ≤ 12 frames.

In [3]:
sync_rows, pooled = G.sync_report()
C1.write_json(C.RESULTS / 'sync_report.json', {'per_match': sync_rows, 'pooled': pooled, 'events_used': P.EVENTS_USED})
print(pd.DataFrame(sync_rows).round(3).to_string(index=False))
print('pooled:', json.dumps(pooled, indent=1))
assert P.EVENTS_USED == pooled['snapped']['accepted'], 'EVENTS_USED must follow the pre-registered sync rule'

 match  events_synced  events_snapped  dropped_unlocated  shot_median_abs_kickoff  shot_median_abs_snapped  shot_p90_abs_snapped  change_median_abs_snapped  agree_agreement_at_lag0  agree_best_lag_frames  agree_agreement_at_best_lag
J03WMX            928             921                  7                     24.5                      1.0                  34.8                        8.0                    0.903                      2                        0.950
J03WN1            624             616                  8                     22.0                      9.0                  96.2                        8.0                    0.881                      3                        0.953
J03WOH            694             657                 37                     44.5                     23.0                  94.0                        7.0                    0.883                      6                        0.950
J03WOY            751             739                 12            

## 3. Label-only statistics (before any model is scored on `L_ctrl`)
Possessions per minute of half time, mean possession duration, share of graded frames changed against the provider labels, agreement with `L_ctrl`.

In [4]:
labels = pd.DataFrame(P.label_table())
labels.to_csv(C.RESULTS / 'label_stats.csv', index=False)
cols = ['possessions', 'possessions_per_min', 'mean_possession_s', 'frames_changed_share', 'agreement_with_ctrl', 'dead_share']
print(labels.pivot(index='match', columns='label_set', values='possessions_per_min').round(2).to_string())
pooled_rates = labels.groupby('label_set')[cols].mean()
pooled_rates['possessions_total'] = labels.groupby('label_set').possessions.sum()
print(); print(pooled_rates.round(4).to_string())
C1.write_json(C.RESULTS / 'label_stats_summary.json', pooled_rates.reset_index().to_dict(orient='records'))

label_set  ctrl  provider  track_1  track_2  track_3
match                                               
J03WMX     3.06      5.25     4.06     3.06     2.48
J03WN1     3.34      5.30     4.21     3.34     2.82
J03WOH     3.05      4.39     3.75     3.05     2.54
J03WOY     3.18      4.80     4.21     3.18     2.61
J03WPY     2.79      4.32     3.53     2.79     2.37
J03WQQ     2.99      4.23     3.78     2.99     2.32
J03WR9     2.73      3.78     3.26     2.73     2.27

           possessions  possessions_per_min  mean_possession_s  frames_changed_share  agreement_with_ctrl  dead_share  possessions_total
label_set                                                                                                                               
ctrl          288.1429               3.0199            11.0288                0.0171               1.0000      0.4488               2017
provider      437.2857               4.5811             7.3400                0.0000               0.9829      

## 4. Re-derivation of the planning findings (cached `exp2_aug`, `target_like`, exp_2's own fold configs)
No fit and no re-selection. Oracles replace dead/alive or team with the provider truth.

In [5]:
oracles = pd.DataFrame(G.oracle_rows())
print(oracles.round(4).to_string(index=False)); print('mean', oracles.drop(columns='match').mean().round(4).to_dict())
misses = G.miss_rows()
miss = G.miss_summary(misses)
print(json.dumps({k: round(v, 4) if isinstance(v, float) else v for k, v in miss.items()}, indent=1))
scored = pd.DataFrame(G.label_score_rows())
table = scored.groupby('label_set')[['spell_f1', 'macro_f1', 'count_error_ratio']].mean()
print(table.round(4).to_string())
C1.write_json(C.RESULTS / 'planning_rederivation.json', {
    'oracles': oracles.to_dict(orient='records'), 'oracle_means': oracles.drop(columns='match').mean().to_dict(),
    'misses': miss, 'exp2_aug_vs_label_sets': table.reset_index().to_dict(orient='records')})

 match  ceiling_5fps  model  oracle_dead_alive  oracle_team_where_both_alive
J03WMX        0.9709 0.6207             0.6462                        0.9200
J03WN1        0.9660 0.6387             0.6628                        0.9177
J03WOH        0.9795 0.6683             0.6852                        0.9378
J03WOY        0.9815 0.6958             0.7126                        0.9431
J03WPY        0.9638 0.6842             0.7082                        0.9230
J03WQQ        0.9840 0.6846             0.7100                        0.9335
J03WR9        0.9668 0.6042             0.6168                        0.9125
mean {'ceiling_5fps': 0.9732, 'model': 0.6566, 'oracle_dead_alive': 0.6774, 'oracle_team_where_both_alive': 0.9268}


{
 "true_possessions": 3061,
 "missed_iou05": 1210,
 "missed_share_under_5s": 0.8661,
 "share_of_possessions_under_2s": 0.3002,
 "recall_under_2s_decoded": 0.284,
 "recall_under_2s_argmax": 0.321,
 "missed_with_ball_visible_ge_80pct": 0.4215,
 "mean_p_true_team_under_2s_ball_visible": 0.603
}
           spell_f1  macro_f1  count_error_ratio
label_set                                       
provider     0.6566    0.9166            -0.1989
track_1      0.6921    0.9172            -0.0470
track_2      0.6797    0.9162             0.2062
track_3      0.6372    0.9117             0.4675


## 5. S2 coverage view tokens
Clean ball re-revealed on one contiguous block of 12 % of each hidden-ball run of the `target_like` view (seeded, label-free).

In [6]:
t0 = time.time()
coverage = pd.DataFrame(P.build_coverage_tokens())
coverage.to_csv(C.RESULTS / 'coverage_stats.csv', index=False)
print(coverage.round(4).to_string(index=False))
print('mean', coverage.drop(columns='match').mean().round(4).to_dict(), f'| {time.time() - t0:.0f}s')
print('exp_3 fits so far:', P.exp3_fit_count(), '| fingerprints:', {a: P.fingerprint(a) for a in P.MODEL_ARMS})
print('exp_2 caches read in place:', P.EXP2_FINGERPRINTS)

 match  hidden_frames  revealed_frames  revealed_share_of_hidden  ball_visible_before  ball_visible_after
J03WMX          90819            10902                    0.1200               0.3778              0.4525
J03WN1          91622            10994                    0.1200               0.3528              0.4304
J03WOH          83335            10001                    0.1200               0.3927              0.4656
J03WOY          88031            10567                    0.1200               0.3824              0.4565
J03WPY          89646            10754                    0.1200               0.3869              0.4604
J03WQQ          75513             9060                    0.1200               0.4695              0.5332
J03WR9          87486            10504                    0.1201               0.4041              0.4756
mean {'hidden_frames': 86636.0, 'revealed_frames': 10397.4286, 'revealed_share_of_hidden': 0.12, 'ball_visible_before': 0.3952, 'ball_visible_after': 0.